### Hybrid Search Langchain

In [ ]:
# !pip install --upgrade --quiet  pinecone pinecone-text pinecone-notebooks

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from langchain_community.retrievers import PineconeHybridSearchRetriever

C:\Users\Akshat Kadia\AppData\Local\Temp\ipykernel_11224\1338964493.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.retrievers import PineconeHybridSearchRetriever


In [3]:
from pinecone import Pinecone, ServerlessSpec
index_name="hybrid-search-langchain-pinecone"
## initialize the Pinecone client
pc=Pinecone(api_key=os.getenv("PINECONE_API_KEY"))

#create the index
if index_name not in pc.list_indexes().names():
    pc.create_index(
        name=index_name,
        dimension=384,  # dimensionality of dense model
        metric="dotproduct",  # sparse values supported only for dotproduct
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )

In [4]:
index=pc.Index(index_name)
index

Index(host='https://hybrid-search-langchain-pinecone-qk0lk8i.svc.aped-4627-b74a.pinecone.io')

In [5]:
## vector embedding and sparse matrix
os.environ["HF_TOKEN"]=os.getenv("HF_TOKEN")

from langchain_huggingface import HuggingFaceEmbeddings
embeddings=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
embeddings

d:\udemy\python\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7194.59it/s]


HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [6]:
from pinecone_text.sparse import BM25Encoder #it uses tf-idf by default

bm25_encoder=BM25Encoder().default()
bm25_encoder

In [7]:
sentences=[
    "Hi, I am Akshat Kadia",
    "I am a B.Tech Graduate from India",
    "I like to learn new things and explore new technologies",
]

## tfidf values on these sentence
bm25_encoder.fit(sentences)

## store the values to a json file
bm25_encoder.dump("bm25_values.json")

# load to your BM25Encoder object
bm25_encoder = BM25Encoder().load("bm25_values.json")


100%|██████████| 3/3 [00:00<00:00, 54.71it/s]


In [8]:
retriever=PineconeHybridSearchRetriever(embeddings=embeddings,sparse_encoder=bm25_encoder,index=index)

In [9]:
retriever

PineconeHybridSearchRetriever(embeddings=HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False), sparse_encoder=<pinecone_text.sparse.bm25_encoder.BM25Encoder object at 0x0000028342E96AD0>, index=Index(host='https://hybrid-search-langchain-pinecone-qk0lk8i.svc.aped-4627-b74a.pinecone.io'))

In [10]:
retriever.add_texts(
    texts=[
        "Hi, I am Akshat Kadia",
        "I am a B.Tech Graduate from India",
        "I like to learn new things and explore new technologies"
]
)

100%|██████████| 1/1 [00:01<00:00,  1.61s/it]


In [12]:
retriever.invoke("What is my Name?")[0].page_content

'Hi, I am Akshat Kadia'